# Current-deck familiarity ablation

This notebook tests one controlled feature addition: `log1p_current_deck_battle_count`. It counts every earlier observed battle in which the player used their current exact eight-card deck, including earlier episodes before switching away and returning.

The player split, optimizer, batch size, early stopping, and three seeds remain unchanged. The same expanded head receives one additional input, changing only its input width from 72 to 73. The only experimental information added is the ninth long-term summary feature.

The original baseline results must already be present in `MyDrive/clash2/data/switch_baseline_expanded_128`. The notebook obtains the current tracked code from GitHub, so only this notebook and the existing Drive data are needed.

In [ ]:
from pathlib import Path

DRIVE_PROJECT_FOLDER = Path("clash2")  # Relative to MyDrive.
REPOSITORY_URL = "https://github.com/jfbami/clash2.git"
MAX_EPOCHS = 60
PATIENCE = 8
MIN_DELTA = 1e-4
BATCH_SIZE = 512
SEEDS = [17, 3407, 918273]


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive") / DRIVE_PROJECT_FOLDER
RAW_BATTLES = DRIVE_PROJECT_ROOT / "data/battles"
BASELINE_RESULTS = DRIVE_PROJECT_ROOT / "data/switch_baseline_expanded_128/results.json"
EXPERIMENT_ROOT = DRIVE_PROJECT_ROOT / "data/switch_current_deck_count_ablation"
DRIVE_CACHE = EXPERIMENT_ROOT / "arrays"
OUTPUT_DIR = EXPERIMENT_ROOT / "training"
required = [RAW_BATTLES, BASELINE_RESULTS]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("Missing required Drive data:\n" + "\n".join(missing))
print(f"Drive project: {DRIVE_PROJECT_ROOT}")
print(f"Experiment output: {EXPERIMENT_ROOT}")


In [ ]:
import shutil
import subprocess

CODE_ROOT = Path("/content/clash2_code")
if CODE_ROOT.exists():
    shutil.rmtree(CODE_ROOT)
subprocess.run(["git", "clone", "--depth", "1", REPOSITORY_URL, str(CODE_ROOT)], check=True)
print(f"Current project code: {CODE_ROOT}")


In [ ]:
import sys

cache_files = [
    DRIVE_CACHE / "metadata.json",
    DRIVE_CACHE / "next_wins.npy",
    DRIVE_CACHE / "propensity_folds.npy",
]
if not all(path.exists() for path in cache_files):
    build_command = [
        sys.executable, "-u", str(CODE_ROOT / "scripts/train_switch_baseline.py"),
        "--cache", str(DRIVE_CACHE),
        "--data-root", str(RAW_BATTLES),
        "--card-reference", str(CODE_ROOT / "data/reference/cards.parquet"),
        "--summary-feature-set", "current_deck_count",
        "--build-cache-only",
    ]
    print("Building the strict experimental cache...")
    subprocess.run(build_command, check=True)
else:
    print("Reusing the existing experimental cache in Drive.")


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Select Runtime → Change runtime type → T4 GPU.")
print(f"GPU: {torch.cuda.get_device_name(0)}")
SESSION_CACHE = Path("/content/clash2_current_deck_count_arrays")
source_metadata = (DRIVE_CACHE / "metadata.json").read_bytes()
cache_is_current = (
    (SESSION_CACHE / "metadata.json").exists()
    and (SESSION_CACHE / "metadata.json").read_bytes() == source_metadata
)
if not cache_is_current:
    if SESSION_CACHE.exists():
        shutil.rmtree(SESSION_CACHE)
    shutil.copytree(DRIVE_CACHE, SESSION_CACHE)
print(f"Session cache ready: {SESSION_CACHE}")


In [ ]:
train_command = [
    sys.executable, "-u", str(CODE_ROOT / "scripts/train_switch_baseline.py"),
    "--cache", str(SESSION_CACHE),
    "--output-dir", str(OUTPUT_DIR),
    "--summary-feature-set", "current_deck_count",
    "--device", "cuda",
    "--max-epochs", str(MAX_EPOCHS),
    "--patience", str(PATIENCE),
    "--min-delta", str(MIN_DELTA),
    "--batch-size", str(BATCH_SIZE),
    "--seeds", *map(str, SEEDS),
]
print("Running:", " ".join(train_command))
subprocess.run(train_command, check=True)


## Compare with the locked baseline

Positive deltas mean the value increased. For AUC, average precision, and accuracy, positive is better. For log loss, Brier score, and calibration error, negative is better.

In [ ]:
import json
import pandas as pd
from IPython.display import display

baseline = json.loads(BASELINE_RESULTS.read_text(encoding="utf-8"))
experiment_results = OUTPUT_DIR / "results.json"
experiment = json.loads(experiment_results.read_text(encoding="utf-8"))
rows = []
for name in baseline["aggregate"]["test"]:
    baseline_metric = baseline["aggregate"]["test"][name]
    experiment_metric = experiment["aggregate"]["test"][name]
    rows.append({
        "metric": name,
        "baseline_mean": baseline_metric["mean"],
        "with_deck_count_mean": experiment_metric["mean"],
        "difference": experiment_metric["mean"] - baseline_metric["mean"],
        "baseline_sd": baseline_metric["sample_standard_deviation"],
        "with_deck_count_sd": experiment_metric["sample_standard_deviation"],
    })
display(pd.DataFrame(rows).set_index("metric").round(6))

per_seed = []
for run in experiment["runs"]:
    per_seed.append({
        "seed": run["seed"],
        "best_epoch": run["best_epoch"],
        "epochs_completed": run["epochs_completed"],
        **{f"test_{key}": value for key, value in run["test"].items()},
    })
display(pd.DataFrame(per_seed).set_index("seed").round(6))


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
for run in experiment["runs"]:
    epochs = [row["epoch"] for row in run["history"]]
    losses = [row["validation_log_loss"] for row in run["history"]]
    plt.plot(epochs, losses, label=f"seed {run['seed']}")
    plt.axvline(run["best_epoch"], linestyle=":", alpha=0.35)
plt.title("Current-deck count: validation log loss")
plt.xlabel("Epoch")
plt.ylabel("Log loss")
plt.grid(alpha=0.25)
plt.legend()
plt.show()


All generated arrays, checkpoints, progress files, and results are stored under `MyDrive/clash2/data/switch_current_deck_count_ablation`. They remain ignored by Git.